# 🫀 실험 10″ — HYP를 쪼개면 Cornell 이야기가 맞는가

**MedKOS / `notebooks/exp10pp_hyp_subclass.ipynb`** · 퀘스트 `ailab-2026-0015`

---

## 이 실험은 **학습을 한 번도 하지 않는다**

실험10′이 남긴 out-of-fold 확률(`arms/fixed_*_f*/probs.npy`)을 **다시 자르기만** 한다.
GPU 불필요, 몇 분이면 끝난다. 그런데도 퀘스트에서 제일 중요한 예측 하나를 판정한다.

## 왜 이걸 하나 — 실험10′이 틀린 예측을 남겼다

사전등록 P2는 이랬다:

> HYP(비대)는 **순수 횡단면 진단**이니 사지유도만 늘려서는 `{12}` 이득의 **1/3 이하**만 얻는다.

실측은 **65%**. 빗나갔다. 원인을 파보니 규칙 카드(`ailab-2026-0016`)가 LVH를
**Sokolow-Lyon 하나**(S V1 + R V5/V6 — 흉부유도 전용)로 대표시킨 것이 오류였다.
실제 LVH 기준 여럿이 **사지유도**를 쓴다:

| 기준 | 유도 | `{I,II}`로 되나 |
|---|---|---|
| Sokolow-Lyon | S V1 + R V5/V6 | ❌ |
| **Cornell voltage** | **R aVL** + S V3 | **앞항만** ✅ |
| Sokolow 사지 변형 | R aVL ≥ 11 mm | ✅ |
| Lewis index | I, III | ✅ |
| Romhilt-Estes 전압 성분 | 사지유도 R/S ≥ 20 mm | ✅ |

`aVL = I − II/2` 이므로 **`{I,II}` 는 aVL을 정확히 갖는다**(실험10′에서 오차 5% 이내 실측).

## 그래서 갈라야 할 것

PTB-XL의 `HYP` superclass는 **한 덩어리가 아니다**. 안에 LVH·RVH·심방확대가 섞여 있다.

- **LVH** — 위 표대로 사지유도 기준이 여럿 → `{I,II}` 비중이 **높아야** 한다
- **RVH** — 주 기준이 **V1의 R/S 비율 > 1**, 우축편위는 사지유도지만 보조 → **낮아야** 한다

**즉 실험10′의 65%는 두 그룹의 평균이라 둘 다 흐렸다.** 쪼개면 갈라져야 한다.

## 사전등록 (결과 보기 전에 고정)

| | 예측 | 근거 |
|---|---|---|
| **G0 검정력 관문** | 소집단 n ≥ 30 인 것만 판정한다. 미만은 **서술만** | HYP 전체가 535건뿐이라 쪼개면 금방 바닥난다 |
| **P-A** | **LVH 의 `{I,II}` 비중 > 65%** | Cornell·Sokolow사지·Lewis·Romhilt 넷이 사지유도를 쓴다 |
| **P-B** | **RVH 의 `{I,II}` 비중 < 33%** | V1의 R/S가 주 기준 = 진짜 횡단면 |
| **P-C** | **비중(LVH) > 비중(RVH)** | 검정력이 모자라 A·B가 각각 안 걸려도, **순서**는 남을 수 있다 |

**세 개가 다 빗나가면 Cornell 해석을 접는다.** 그 경우 65%는 다른 이유(예: HYP 환자의
동반 소견이 사지유도에 보이는 것)로 설명해야 하고, 웨어러블 LVH 스크리닝 주장은 철회한다.

**판정은 3분한다 — 지지 / 기각 / 미결.** CI가 임계값을 걸치면 ❌가 아니라 ⚠️ 미결이다.
검정력 부족을 반증으로 위장하지 않기 위해서다. 합성 픽스처로 확인한 바로는 **n≈40이면
비중 CI 폭이 1.0 근처**라, RVH가 드물 경우 P-B는 미결로 끝날 가능성이 높다 —
그것도 결과다(다음 데이터셋을 정하는 정보).

## 지표를 F1이 아니라 **AUC**로 잡는 이유

실험10′의 헤드라인은 동작점 정합 F1이었다. 그런데 여기선 소집단이 수십 건이라
**임계값을 넘긴 개수**를 세면 한두 건에 결과가 출렁인다. 그래서 주지표를 바꾼다:

> **소집단 AUC** = 소집단 환자의 HYP 확률이 무작위 NORM 환자보다 높을 확률

순위 통계라 임계값이 없고, 같은 표본에서 F1보다 검정력이 훨씬 높다.
**F1(동작점 정합)도 같이 보고**한다 — 실험10′ 헤드라인과 잇기 위해서다.
지표를 바꾼 것은 **결과를 보기 전** 이 칸에 적어둔다.


In [ ]:
# CELL 1 — 설정 (학습 없음 · 실험10′ 산출물 재사용)
!pip -q install wfdb

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★ 실험10′ CELL 1과 동일해야 하는 값(정렬 보존용)
CLASSES = ["NORM", "CD", "STTC", "MI", "HYP"]
CONFIGS = {"II": [1], "I+II": [0, 1], "II+V1": [1, 6], "12": list(range(12))}
K_FOLD  = 5
SEED0   = 20260801
BOOT    = 4000          # 소집단이라 재표본을 넉넉히
NMIN    = 30            # G0 검정력 관문

# HYP superclass 안의 소집단 (PTB-XL diagnostic_subclass)
SUBGROUPS = {
    "LVH":      ["LVH"],            # 좌심실비대 — 사지유도 기준 다수
    "RVH":      ["RVH"],            # 우심실비대 — V1 R/S 가 주 기준
    "심방확대": ["LAO/LAE", "RAO/RAE"],
    "SEHYP":    ["SEHYP"],          # 중격비대
}

CONFIG = dict(exp="exp10pp_hyp_subclass", quest="ailab-2026-0015",
              parent_exp="exp10p_lead_cv", training="없음(재분석)",
              question="HYP를 LVH/RVH로 쪼개면 {I,II} 비중이 갈리는가",
              primary_metric="소집단 AUC(HYP 확률, NORM 대조) 기준 {I,II} 이득 비중",
              secondary_metric="동작점 정합 F1 기준 같은 비중(실험10′ 헤드라인과 연결)",
              predictions={"G0": f"소집단 n>={NMIN} 인 것만 판정",
                           "P-A": "LVH 비중 > 0.65",
                           "P-B": "RVH 비중 < 0.33",
                           "P-C": "비중(LVH) > 비중(RVH)"},
              rationale="aVL = I - II/2 이므로 {I,II}는 Cornell voltage의 앞항(R aVL)을 갖는다",
              subgroups=SUBGROUPS, k_fold=K_FOLD, boot=BOOT, seed0=SEED0)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp10pp_hyp_subclass", CONFIG, project=PROJECT)

REG = os.path.join(PROJECT, "registry.jsonl")
prev = None
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if r.get("exp_id") == "exp10p_lead_cv" and os.path.isdir(r.get("dir", "")):
        prev = r
if prev is None:
    raise RuntimeError("registry.jsonl에서 exp10p_lead_cv를 못 찾았습니다")
PREV = prev["dir"]
run.log(f"실험10′ 산출물: {PREV}")

def prev_arm(name):
    p = os.path.join(PREV, "arms", name, "probs.npy")
    return np.load(p) if os.path.exists(p) else None

missing = [f"fixed_{c}_f{k}" for c in CONFIGS for k in range(K_FOLD)
           if prev_arm(f"fixed_{c}_f{k}") is None]
if missing:
    raise RuntimeError(f"실험10′ arm 없음: {missing[:5]} … 총 {len(missing)}개")
run.log(f"✅ 유도고정 arm {len(CONFIGS)*K_FOLD}개 확인 — 학습은 하지 않습니다")

In [ ]:
# CELL 2 — 라벨만 읽는다 (X는 건드리지 않는다 → 390MB 압축해제 회피)
import pandas as pd, subprocess

PTB = "/content/ptbxl"
os.makedirs(PTB, exist_ok=True)
BASE = "https://physionet.org/files/ptb-xl/1.0.3"
for f in ("ptbxl_database.csv", "scp_statements.csv"):
    d = os.path.join(PTB, f)
    if not (os.path.exists(d) and os.path.getsize(d) > 0):
        subprocess.run(["wget", "-q", "-O", d, f"{BASE}/{f}"])
df  = pd.read_csv(os.path.join(PTB, "ptbxl_database.csv"), index_col="ecg_id")
scp = pd.read_csv(os.path.join(PTB, "scp_statements.csv"), index_col=0)

agg = scp[scp.diagnostic == 1].diagnostic_class.to_dict()
sub_of = scp[scp.diagnostic == 1].diagnostic_subclass.to_dict()
df["sc"] = df.scp_codes.apply(
    lambda s: sorted({agg[k] for k in ast.literal_eval(s) if k in agg}))
sub = df[df.sc.apply(lambda s: len(s) == 1 and s[0] in CLASSES)].copy()

CACHE = run.data("ptbxl_12lead_full.npz")
if not os.path.exists(CACHE):
    raise RuntimeError(f"캐시가 없습니다: {CACHE}")
z = np.load(CACHE, allow_pickle=True)
Y, FOLD10, EID = z["y"], z["fold"], z["eid"]      # ★ z["X"] 를 읽지 않는다
if len(Y) != len(sub) or not np.array_equal(np.sort(EID), np.sort(sub.index.values)):
    raise RuntimeError("캐시가 지금 sub와 다르다 — 실험10′과 같은 캐시가 맞는지 확인")
sub = sub.loc[EID]                                 # OOF 행 순서에 맞춘다
CV  = (FOLD10 - 1) % K_FOLD
run.log(f"레코드 {len(Y):,} · 클래스 {np.bincount(Y, minlength=5).tolist()} ({CLASSES})")

# HYP 레코드 각각이 가진 diagnostic_subclass 집합
sub["subcls"] = sub.scp_codes.apply(
    lambda s: sorted({sub_of[k] for k in ast.literal_eval(s) if k in sub_of}))
HYP_I = CLASSES.index("HYP")
hyp_m = (Y == HYP_I)
run.log(f"HYP {hyp_m.sum()}건")

# OOF 확률 복원 (겹별 arm → 전체 배열)
OOF = {}
for c in CONFIGS:
    arr = np.zeros((len(Y), len(CLASSES)))
    for k in range(K_FOLD):
        arr[np.where(CV == k)[0]] = prev_arm(f"fixed_{c}_f{k}")
    OOF[c] = arr
run.log("OOF 확률 복원 완료 (유도고정 4구성)")

### CELL 3 — 【G0 검정력 관문】 쪼갤 수 있는가부터

535건을 넷으로 나누면 몇 건씩 남는지부터 센다. **LLM이 추측하지 않고 여기서 실측한다.**
`n < 30` 인 소집단은 판정 대상에서 빼고 서술만 한다 — 없는 검정력을 있는 척하지 않기 위해서다.

`순수`는 그 소집단 코드만 가진 레코드, `포함`은 다른 HYP 코드도 같이 가진 레코드까지다.
**주분석은 `순수`** 로 한다(LVH+RVH 동반이면 어느 쪽 기전인지 못 가른다).


In [ ]:
# CELL 3 — 소집단 실현가능성
hyp_idx = np.where(hyp_m)[0]
codes_all = sorted({c for i in hyp_idx for c in sub.subcls.iloc[i]})
run.log(f"HYP 안에서 관측된 subclass 코드: {codes_all}")

MEMB, feas = {}, {}
for name, codes in SUBGROUPS.items():
    inc  = np.array([bool(set(sub.subcls.iloc[i]) & set(codes)) for i in hyp_idx])
    pure = np.array([set(sub.subcls.iloc[i]) <= set(codes) and len(sub.subcls.iloc[i]) > 0
                     for i in hyp_idx]) & inc
    MEMB[name] = {"포함": hyp_idx[inc], "순수": hyp_idx[pure]}
    feas[name] = {"포함": int(inc.sum()), "순수": int(pure.sum())}

run.log("\n【G0】 소집단 표본")
run.log(f"  {'소집단':<10}{'순수':>8}{'포함':>8}   판정")
TESTABLE = []
for name in SUBGROUPS:
    n = feas[name]["순수"]
    ok = n >= NMIN
    if ok: TESTABLE.append(name)
    run.log(f"  {name:<10}{n:>8}{feas[name]['포함']:>8}   "
            + ("✅ 판정" if ok else f"⚠️ 서술만 (n<{NMIN})"))
run.log(f"\n판정 가능: {TESTABLE if TESTABLE else '없음'}")
if "LVH" not in TESTABLE:
    run.log("⛔ LVH조차 표본이 안 된다 — P-A/P-C 는 이번 실험으로 판정 불가")

### CELL 4 — 소집단별 유도 이득

각 소집단에 대해 네 구성의 **AUC**(그 소집단 vs 전체 NORM, HYP 확률 기준)를 재고,

```
비중 = [AUC(I+II) − AUC(II)] / [AUC(12) − AUC(II)]
```

를 구한다. **분모가 0을 걸치면 비중은 정의되지 않는다** — 그럴 땐 비중 대신 두 Δ를 따로
보고한다(0으로 나누기를 "결과"로 위장하지 않기).

부트스트랩은 **소집단 환자와 NORM 대조를 각각 재표본**한다.


In [ ]:
# CELL 4 — 소집단별 AUC · 비중
from sklearn.metrics import roc_auc_score, f1_score

norm_idx = np.where(Y == 0)[0]

def auc_of(cfg, gidx, gi=None, ni=None):
    """소집단(gidx) vs NORM 의 HYP 확률 AUC. gi/ni 는 부트스트랩 재표본 인덱스."""
    g = gidx if gi is None else gidx[gi]
    n = norm_idx if ni is None else norm_idx[ni]
    s = np.concatenate([OOF[cfg][g, HYP_I], OOF[cfg][n, HYP_I]])
    y = np.concatenate([np.ones(len(g)), np.zeros(len(n))])
    return float(roc_auc_score(y, s))

# 실험10′과 같은 동작점 정합 (F1 부지표용)
def alpha_for(prob, target):
    lo, hi = 0.02, 50.0
    for _ in range(40):
        mid = (lo * hi) ** 0.5
        p = prob.copy(); p[:, 0] *= mid
        if float((p.argmax(1)[Y == 0] != 0).mean()) > target: lo = mid
        else: hi = mid
    return hi

ref_fa = float((OOF["II"].argmax(1)[Y == 0] != 0).mean())
PRED = {}
for c in CONFIGS:
    p = OOF[c].copy(); p[:, 0] *= alpha_for(OOF[c], ref_fa)
    PRED[c] = p.argmax(1)
run.log(f"기준 오경보율 = 유도고정 {{II}}의 {ref_fa:.3f} (실험10′과 동일 절차)")

rs = np.random.RandomState(SEED0)
RESULT = {}
run.log("\n" + "=" * 92)
run.log("【표】 소집단별 AUC (HYP 확률 · NORM 대조)")
run.log("=" * 92)
run.log(f"  {'소집단':<10}{'n':>5}" + "".join(f"{c:>10}" for c in CONFIGS)
        + f"{'Δ(I+II)':>10}{'Δ(12)':>9}{'비중':>18}")

for name in SUBGROUPS:
    g = MEMB[name]["순수"]
    if len(g) == 0:
        run.log(f"  {name:<10}{0:>5}   (해당 없음)"); continue
    a = {c: auc_of(c, g) for c in CONFIGS}
    d_i2, d_12 = a["I+II"] - a["II"], a["12"] - a["II"]

    bs_num, bs_den, bs_share = [], [], []
    for _ in range(BOOT):
        gi = rs.randint(0, len(g), len(g))
        ni = rs.randint(0, len(norm_idx), len(norm_idx))
        b = {c: auc_of(c, g, gi, ni) for c in ("II", "I+II", "12")}
        num, den = b["I+II"] - b["II"], b["12"] - b["II"]
        bs_num.append(num); bs_den.append(den)
        if den > 1e-6: bs_share.append(num / den)
    q = lambda v, p: float(np.percentile(v, p)) if len(v) else float("nan")
    den_ci = (q(bs_den, 2.5), q(bs_den, 97.5))
    den_ok = den_ci[0] > 0                      # 12유도가 유의하게 이득을 줘야 비율이 뜻을 가진다
    share = d_i2 / d_12 if abs(d_12) > 1e-6 else float("nan")
    sh_ci = (q(bs_share, 2.5), q(bs_share, 97.5)) if den_ok else (float("nan"),) * 2

    # 부지표: 동작점 정합 F1(그 소집단의 HYP 재현율)
    rec = {c: float((PRED[c][g] == HYP_I).mean()) for c in CONFIGS}

    RESULT[name] = {"n_pure": int(len(g)), "n_incl": int(len(MEMB[name]["포함"])),
                    "auc": a, "delta_i2": d_i2, "delta_12": d_12,
                    "delta_i2_ci": [q(bs_num, 2.5), q(bs_num, 97.5)],
                    "delta_12_ci": list(den_ci), "denominator_positive": bool(den_ok),
                    "share": None if not den_ok else share,
                    "share_ci": None if not den_ok else list(sh_ci),
                    "recall_matched": rec, "testable": name in TESTABLE}
    sh_s = (f"{share:>7.1%} [{sh_ci[0]:.0%},{sh_ci[1]:.0%}]" if den_ok
            else "     정의불가(분모 CI가 0 포함)")
    run.log(f"  {name:<10}{len(g):>5}" + "".join(f"{a[c]:>10.3f}" for c in CONFIGS)
            + f"{d_i2:>+10.3f}{d_12:>+9.3f}{sh_s:>18}"
            + ("" if name in TESTABLE else "   ⚠️서술만"))

run.log("\n【부지표】 동작점 정합 HYP 재현율 (실험10′ 헤드라인과 연결)")
run.log(f"  {'소집단':<10}" + "".join(f"{c:>10}" for c in CONFIGS))
for name, r in RESULT.items():
    run.log(f"  {name:<10}" + "".join(f"{r['recall_matched'][c]:>10.3f}" for c in CONFIGS))

In [ ]:
# CELL 5 — 사전등록 채점
def sh(name):
    r = RESULT.get(name)
    return None if not r or r["share"] is None else r["share"]

run.log("\n" + "=" * 92)
run.log("【사전등록 채점】")
run.log("=" * 92)

PA = PB = PC = None
notes = []

def decide(name, thr, direction):
    """3분: 지지(True) / 기각(False) / 미결(None).
    CI가 임계값을 걸치면 '기각'이 아니라 '미결'이다 — 검정력 부족을 반증으로 위장하지 않는다."""
    r = RESULT.get(name)
    if not r or r["share"] is None or name not in TESTABLE:
        return None, "표본 또는 분모 부족"
    lo, hi = r["share_ci"]
    if direction == ">":
        if lo > thr: return True,  f"CI 하한 {lo:.1%} > {thr:.0%}"
        if hi < thr: return False, f"CI 상한 {hi:.1%} < {thr:.0%}"
    else:
        if hi < thr: return True,  f"CI 상한 {hi:.1%} < {thr:.0%}"
        if lo > thr: return False, f"CI 하한 {lo:.1%} > {thr:.0%}"
    return None, f"CI [{lo:.0%},{hi:.0%}] 가 {thr:.0%}를 걸침 — 검정력 부족"

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}
PA, why = decide("LVH", 0.65, ">")
run.log(f"  P-A LVH 비중 > 65% → {MARK[PA]} "
        + (f"({sh('LVH'):.1%}, {why})" if sh("LVH") is not None else f"({why})"))
if PA is None: notes.append(f"P-A 미결: {why}")

PB, why = decide("RVH", 0.33, "<")
run.log(f"  P-B RVH 비중 < 33% → {MARK[PB]} "
        + (f"({sh('RVH'):.1%}, {why})" if sh("RVH") is not None else f"({why})"))
if PB is None:
    notes.append(f"P-B 미결: {why}")
    run.log("      (픽스처 검증상 n≈40이면 CI 폭이 1.0 근처다 — RVH 미결은 예상된 결과이고, "
            "'RVH 표본이 없다'가 곧 다음 데이터셋을 정하는 정보다)")

if sh("LVH") is not None and sh("RVH") is not None:
    PC = bool(sh("LVH") > sh("RVH"))
    run.log(f"  P-C 비중(LVH) > 비중(RVH) → {'✅' if PC else '❌'} "
            f"({sh('LVH'):.1%} vs {sh('RVH'):.1%})")
else:
    run.log("  P-C → ⚠️ 판정불가")

if PA is None and PB is None:
    verdict = ("판정불가(검정력) — HYP 535건을 쪼개니 소집단이 남지 않는다. "
               "Cornell 해석은 확인도 반증도 못 했다. 다른 데이터셋(Chapman/Ningbo는 "
               "LVH·RVH 라벨이 별도)으로 옮겨야 한다")
elif PA:
    verdict = (f"Cornell 해석 지지 — LVH의 {{I,II}} 비중 {sh('LVH'):.1%} 로 "
               "HYP 전체(65%)보다 높다. 실험10′의 65%는 RVH가 섞여 희석된 값이었다. "
               "웨어러블 LVH **스크리닝** 주장의 근거가 선다(진단 아님)")
elif PA is False:
    verdict = (f"Cornell 해석 기각 — LVH만 재도 비중이 {sh('LVH'):.1%} 로 65%를 못 넘는다. "
               "65%는 사지유도 LVH 기준 때문이 아니라 다른 이유(동반 소견 등)다. "
               "`ailab-2026-0016`의 Cornell 근거를 약화 표기하고 웨어러블 주장은 철회한다")
else:
    verdict = "부분 — 아래 표로 서술"
run.log(f"\n▶ {verdict}")
if notes: run.log("  단서: " + " · ".join(notes))
run.log("=" * 92)

import matplotlib.pyplot as plt
plot = [n for n in RESULT if RESULT[n]["n_pure"] > 0]
fig, ax = plt.subplots(figsize=(7, 3.8))
xs = np.arange(len(plot))
for i, c in enumerate(CONFIGS):
    ax.bar(xs + i * 0.21 - 0.32, [RESULT[n]["auc"][c] for n in plot], 0.2, label=c)
ax.set_xticks(xs)
ax.set_xticklabels([f"{n}\n(n={RESULT[n]['n_pure']})" for n in plot])
ax.set_ylim(0.4, 1.0); ax.axhline(0.5, c="k", lw=0.8, ls=":")
ax.set_ylabel("AUC (HYP 확률 · NORM 대조)")
ax.set_title("실험10″ — HYP 소집단별 유도 구성 이득")
ax.legend(fontsize=8, ncol=4); plt.tight_layout()
run.save_fig("hyp_subclass_auc", fig); plt.show()

run.save_json("evaluation", {"feasibility": feas, "result": RESULT,
                             "testable": TESTABLE, "P-A": PA, "P-B": PB, "P-C": PC,
                             "verdict": verdict})

result = {"week": 2, "exp_id": "exp10pp_hyp_subclass", "quest": "ailab-2026-0015",
          "task": "HYP를 subclass로 쪼개면 {I,II} 비중이 LVH/RVH로 갈리는가",
          "split": "inter", "metric": "share_i2_of_12_gain_LVH",
          "value": None if sh("LVH") is None else round(sh("LVH"), 4),
          "passed": bool(PA), "date": time.strftime("%Y-%m-%d"),
          "training": "없음(실험10′ OOF 재분석)", "n_hyp": int(hyp_m.sum()),
          "feasibility": feas, "subgroups": RESULT, "testable": TESTABLE,
          "predictions": {"P-A": PA, "P-B": PB, "P-C": PC}, "verdict": verdict,
          "summary": (f"LVH n={feas['LVH']['순수']} 비중="
                      + ("판정불가" if sh("LVH") is None else f"{sh('LVH'):.1%}")
                      + f" · RVH n={feas['RVH']['순수']} 비중="
                      + ("판정불가" if sh("RVH") is None else f"{sh('RVH'):.1%}")
                      + f" · {verdict.split(' —')[0]}")}
result = run.finish(result)
import shutil; shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")
print(f"""
────────────────────────────────────────────────────────────────
📁 {run.dir}
  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp10pp_hyp_subclass.ipynb \\
      --quest ailab-2026-0015 --step "exp10pp-hyp-subclass" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")

---

## 결과 읽는 법

| P-A | P-B | 뜻 | 다음 |
|---|---|---|---|
| ✅ | ✅ | **완전 확증.** LVH는 사지유도가 많이 열고 RVH는 안 연다 — 기전까지 규명 | `ailab-2026-0016` 2-bis를 subclass 단위로 재작성. 웨어러블 LVH 스크리닝을 실험11의 목표로 승격 |
| ✅ | ⚠️ | LVH는 확인, RVH는 표본 부족 | 절반의 확증. RVH는 Chapman/Ningbo로 넘긴다 |
| ❌ | — | Cornell 해석이 틀렸다 | **카드를 되돌린다.** 65%의 원인을 다시 찾는다(동반 소견? 라벨 잡음?) |
| ⚠️ | ⚠️ | 쪼갤 표본이 없다 | 이 데이터셋으로는 끝. 다른 데이터셋으로 이동 |

**⚠️ 두 개가 나와도 실망할 결과가 아니다.** "535건을 쪼개면 판정이 안 된다"는 것은
**다음 데이터셋 선택을 결정하는 정보**다(차별점 A의 표를 채우려면 subclass 라벨이 큰
데이터셋이 필요하다는 뜻).

## 한계 — 먼저 적어둔다

- **모델은 subclass를 배운 적이 없다.** 5-superclass 분류기이므로, 여기서 재는 것은
  "HYP 확률이 소집단마다 얼마나 잘 오르나"이지 subclass 진단 성능이 아니다.
  → LVH 비중이 높게 나와도 그건 "**사지유도가 LVH 신호를 더 실어나른다**"는 뜻이지
  "LVH를 진단할 수 있다"가 아니다.
- **AUC는 F1보다 관대하다.** 순위만 보므로 임상 동작점에서의 쓸모와 다르다.
  그래서 재현율 표를 같이 낸다 — 둘의 방향이 어긋나면 AUC 쪽을 믿지 않는다.
- **PTB-XL의 subclass 라벨도 자동판독기 유래다.** LVH 라벨 자체가 Sokolow/Cornell 같은
  전압 기준으로 붙었을 수 있다 → **순환논리 위험**. 이 실험이 "지지"로 나오면
  라벨 생성 절차를 반드시 확인하고, 확인 전까지 결론은 잠정으로 둔다.
